# M4: Working with Files — in-class notebook (SOLUTIONS)

**File you need (in this folder):** `names.txt`
Sync your fork first so you have it!

**Part 1** walks through the in-class exercise: building a directory of buildings where MACSS classes meet.
**Part 2**, the Nobel Prize application, is in its own notebook: `M4_files_nobel.ipynb`.

**Reading:** [CS with Applications §4.1, Working with Files](https://book.cs-apps.org/working_with_data/files/index.html). Each step below links to the section it practices.

**How to use this notebook:**
- **Predict** cells: write your guess in a comment *before* you run the cell.
- **Your turn** cells: fill in `# YOUR CODE HERE`.
- **Check** cells: run them after your code. Silence means it worked; an `AssertionError` means try again.
- **Discuss** prompts: talk it through with your partner.

Remember the common programming pattern:
1. **Load** the data from disk into a data structure
2. **Manipulate** the data
3. **Write** the result (print it, or write it back to disk)

---
# Part 1: Buildings directory

You are working on a project creating a directory of buildings based on where MACSS classes typically meet.
`names.txt` is the university's building list: each building is an abbreviation, a **tab**, then the name and address.

## Step 1: Open, read, close
**Reading:** [§4.1.1 Basic file I/O](https://book.cs-apps.org/working_with_data/files/index.html#basic-file-i-o)

The most basic version: open the file, read the *whole thing* into one string, close it.

- `open()` doesn't read anything yet. It gives you a **file object**, which keeps track of *where you are* in the file.
- `.read()` reads from where you are to the end, and returns one big string.
- `.close()` releases the file. Forgetting it is easy, which is why we'll switch to `with` in Step 2.

In [ ]:
f = open("names.txt")
contents = f.read()
f.close()

print(type(contents))
print(contents[:300])

**Predict:** what does `len(contents)` tell us? Is it the number of buildings?

In [ ]:
# My prediction:
len(contents)

It's the number of **characters**: `contents` is one string, not a list of lines.

**Predict:** what happens if you call `.read()` twice on the same file object?

In [ ]:
# My prediction:
with open("names.txt") as f:
    first = f.read()
    second = f.read()

print(len(first))
print(repr(second))

The first `read()` moves the file's position to the end, so the second has nothing left and returns `''` (the reading makes the same point). To read it again, open the file again.

## Step 2: Use `with` and read line by line
**Reading:** [§4.1.1 Basic file I/O](https://book.cs-apps.org/working_with_data/files/index.html#basic-file-i-o)

`with open(...) as f:` closes the file for us when the indented block ends, even if there's an error. From now on, use `with`.

Looping over the file (`for line in f`) gives one line at a time, so you never hold the whole file in memory. That matters for big files.

**Predict:** why is there a blank line between each printed line?

In [ ]:
with open("names.txt") as f:
    for i, line in enumerate(f):
        print(line)
        if i == 6:
            break

Each `line` ends with `"\n"`, and `print` adds *another* newline. `repr()` shows what's *really* in a string: hidden spaces, tabs (`\t`), and newlines (`\n`). It's your best debugging tool for files.

**Discuss:** look at the output below. What separates the abbreviation from the name? What else do you notice at the start of most lines?

In [ ]:
with open("names.txt") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i == 6:
            break

## Step 3: Load the data into a data structure

This is step 1 of the pattern: get the data off disk and into a structure you can work with. A list of `[abbreviation, name]` pairs lets us look at each part separately.

**Your turn:** build a list `buildings` where each element is a list `[abbreviation, name_and_address]`.

Hints:
- `line.strip()` removes whitespace (including `\n`) from both ends
- `line.split("\t", 1)` splits on the **first** tab only

In [ ]:
buildings = []
with open("names.txt") as f:
    for line in f:
        builds = line.strip().split("\t", 1)
        buildings.append(builds)

buildings[:8]

In [ ]:
# Check
assert len(buildings) == 152, "one entry per line in the file"
assert buildings[4] == ["1155", "1155 E 60th St"]
assert ["SS", " Social Sciences Research Building, 1126 E. 59th St."] in buildings,     "did you strip each line before splitting?"


Look at the first few rows. The file has a header (title, column names, a blank line) that isn't data.
How many lines? Look at the end of the list too.

In [ ]:
print(len(buildings))
buildings[-4:]

### What if you want to skip lines?

Two approaches from the slides: call `next()` on the file to skip lines, or use `enumerate` and keep only certain line numbers.

**Discuss:** which approach would still work if the registrar added one more line to the header?

In [ ]:
# Approach 1: next() skips one line each time
with open("names.txt") as f:
    next(f)
    next(f)
    next(f)
    next(f)
    first_data_line = f.readline()

repr(first_data_line)

In [ ]:
# Approach 2: enumerate. Keep only the lines that are buildings.
# Hint: which line index is the first building? Which is the last?
data_lines = []
with open("names.txt") as f:
    for i, line in enumerate(f):
        if line.strip() and 4 <= i and "\t" in line:
            data_lines.append(line)

print(repr(data_lines[0]))
print(repr(data_lines[-1]))

In [ ]:
# Check
assert data_lines[0].startswith("1155")
assert "ZDC" in data_lines[-1]

## Step 4: Manipulate — select the buildings we need

Step 2 of the pattern: now that the data is in a list, we can filter it. We want these buildings:

In [ ]:
sched = ["1155", "SS", "TTI", "K", "STU"]

**Predict:** this looks right. How many of the 5 buildings will it find? Vote: 0, 1, 5, or an error.

In [ ]:
# My prediction:
buildings_select = []
with open("names.txt") as f:
    for line in f:
        abbrev = line.split("\t", 1)[0]
        if abbrev in sched:
            buildings_select.append(abbrev)

buildings_select

Only one match! Look at `repr()` of the abbreviation for Social Sciences. What's going on?

In [ ]:
with open("names.txt") as f:
    for line in f:
        if "Social Sciences" in line:
            print(repr(line))

Most abbreviations have a **leading space** (`" SS"`), so `" SS" in sched` is `False`. Python compares strings character by character, and a space is a character.

The lesson: **always look at your raw data with `repr()`** before you trust a comparison.

**Your turn:** using your `buildings` list from Step 3, build `buildings_select`: the `[abbrev, name]` pairs whose abbreviation is in `sched`. Do you get all 5?

In [ ]:
buildings_select = []
for line in buildings:
    if line[0].strip() in sched:
        buildings_select.append(line)

buildings_select

In [ ]:
# Check
assert len(buildings_select) == 5
assert [b[0].strip() for b in buildings_select] == ["1155", "K", "SS", "STU", "TTI"]

## Step 5: Write the result to a file
**Reading:** [§4.1.2 Writing data to a file](https://book.cs-apps.org/working_with_data/files/index.html#writing-data-to-a-file)

Step 3 of the pattern. Open the file in **write mode** (`"w"`). `print(..., file=f)` handles the newline for you; `f.write(s)` doesn't, so you'd add `"\n"` yourself.

| mode | what it does |
|---|---|
| `"r"` | read (the default) |
| `"w"` | write: **creates** the file, or **wipes** it if it exists |
| `"a"` | append: adds to the end |

**Predict:** what will `scratch.txt` contain after this cell runs?

In [ ]:
# My prediction:
with open("scratch.txt", "w") as f:
    print("first", file=f)

with open("scratch.txt", "w") as f:
    print("second", file=f)

with open("scratch.txt", "a") as f:
    print("third", file=f)

with open("scratch.txt") as f:
    print(f.read())

Opening in `"w"` wiped `first`. **Be careful:** `"w"` on the wrong file name will erase your data with no warning.

In [ ]:
# Write each selected building to selected_buildings.txt, one per line
with open("selected_buildings.txt", "w") as f:
    for build in buildings_select:
        print(build, file=f)

Check your work by reading the file back in:

In [ ]:
with open("selected_buildings.txt") as f:
    print(f.read())

Each line is the *printed list* (`['SS', ' Social Sciences...']`). Not very readable!
**Your turn:** write a cleaner version, `selected_buildings_clean.txt`, with lines like `SS: Social Sciences Research Building, 1126 E. 59th St.`

In [ ]:
with open("selected_buildings_clean.txt", "w") as f:
    for abbrev, name in buildings_select:
        print(f"{abbrev.strip()}: {name.strip()}", file=f)

with open("selected_buildings_clean.txt") as f:
    print(f.read())

## Step 6: Bringing it together in a function
**Reading:** [§4.1.2](https://book.cs-apps.org/working_with_data/files/index.html#writing-data-to-a-file) does the same thing with `strip_domain`: load → transform → write, each as its own block.

Why a function? Next quarter the schedule changes, so you call it with a new `sched`. A new campus file? Pass a new `input_filename`. Nothing inside has to change.

**Your turn:** fill in `get_buildings` so it works for any input file, output file, and schedule.

In [ ]:
def get_buildings(input_filename, output_filename, sched):
    '''
    Extract the buildings in sched from the campus list
    and write them to output_filename, one per line.

    Inputs:
      input_filename: (string) name of a file with buildings
      output_filename: (string) name for the output file
      sched: (list of strings) building abbreviations to keep

    Returns: (list of lists) the selected [abbrev, name] pairs
    '''
    # Load data into a data structure (a list of lists)
    buildings = []
    with open(input_filename) as f:
        for line in f:
            builds = line.strip().split("\t", 1)
            buildings.append(builds)

    # Transform the data
    buildings_select = []
    for line in buildings:
        if line[0] in sched:
            buildings_select.append(line)

    # Write the data
    with open(output_filename, "w") as f:
        for build in buildings_select:
            print(build, file=f)

    return buildings_select

In [ ]:
result = get_buildings("names.txt", "selected_buildings.txt", sched)
assert len(result) == 5, f"expected 5 buildings, got {len(result)}"
result

**Discuss:** in the function, `line[0] in sched` works *without* `.strip()` on the abbreviation. Why? (Hint: what does `line.strip()` do to `" SS\t Social..."`?)

## Exam prep: spot 3 errors

This version of `get_buildings` has **3 errors**. Find them *on paper* first (that's how the exam works), then write them down in the next cell.

For each error, also decide: would Python give you an error message, or would it run and give a wrong answer?

```python
def get_buildings(input_filename, output_filename, sched):
    buildings = []
    with open(input_filename) as f:
        for line in f:
            builds = line.strip().split("\t", 1)
            buildings.append(builds)

    buildings_select = []
    for line[0] in buildings:
        if line in sched:
            buildings_select.append(line)

    with open(output_filename) as f:
        for build in buildings:
            print(build, file=f)
```

In [ ]:
# Your errors:
# 1. `for line[0] in buildings:` - the loop variable must be a name: `for line in buildings:`
#    (then compare `line[0] in sched`, not `line in sched`)
#    Python catches it: `line` is still a string from the first loop, so
#    `line[0] = ...` raises TypeError: 'str' object does not support item assignment
# 2. `open(output_filename)` opens in read mode, so print(..., file=f) fails. Needs "w".
#    Python catches it: io.UnsupportedOperation: not writable
# 3. `for build in buildings:` writes ALL buildings. Should loop over buildings_select.
#    Python does NOT catch it: it runs and writes the wrong file. Test your output!